# MM-WLAuslan 3D keypoint extraction

这个 notebook 专门从 MM-WLAuslan 数据重新生成 `keypoints3d/`。
默认支持 zip 数据集和散装视频，并且可以断点续跑；不会处理 Auslan-Daily 的 Excel/TXT 标注文件。

如果只想修正指定单词，把 `ONLY_WORDS` 改成单词列表；保持 `None` 会处理 Drive 文件夹里的全部视频。

In [ ]:
!pip -q install 'mediapipe==0.10.33' opencv-python-headless numpy scipy gdown google-api-python-client

In [ ]:
from pathlib import Path

DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1iRb1b_0JQEUVnC2EldfkXjaWoUNWQ0Dv'
CODE_FOLDER_URL = 'https://drive.google.com/drive/folders/1f03erNchIoWPc9zevCBHJz0n46dHYM34'
REPO_URL = 'https://github.com/randlyoyo/FIT5120-TE38-SignLanguage.git'
PROJECT = Path('/content/signtest')
DRIVE_CODE_ROOT = Path('/content/drive/MyDrive/signtest_code')
DATA_ROOT = Path('/content/drive_video_source')
OUTPUT = PROJECT / 'keypoints3d'

# None = all videos. 例如只处理一批词时可改成 ['BRACELET', 'BREAD'].
ONLY_WORDS = None

# A100 使用 GPU delegate；先用 1 个 worker 做 EGL 冒烟测试。
USE_GPU = True
WORKERS = 1
GPU_SMOKE_TEST = True

print('PROJECT:', PROJECT)
print('DATA_ROOT:', DATA_ROOT)
print('ONLY_WORDS:', 'all' if ONLY_WORDS is None else len(ONLY_WORDS))

In [ ]:
# 实验性 A100/EGL 配置：MediaPipe GPU 需要无显示器 EGL/OpenGL 上下文。
import os
import subprocess

if USE_GPU:
    egl_packages = ['libegl1', 'libgles2', 'libgl1', 'libegl1-mesa-dev', 'libgles2-mesa-dev']
    print('installing EGL packages:', ' '.join(egl_packages))
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', *egl_packages], check=True)
    os.environ['EGL_PLATFORM'] = 'surfaceless'
    os.environ['GLOG_logtostderr'] = '1'
    print('EGL_PLATFORM:', os.environ['EGL_PLATFORM'])
else:
    print('USE_GPU=False; skip EGL setup')

In [ ]:
import shutil
import subprocess
import sys
import urllib.request
import zipfile

required_code = [PROJECT / 'extract3d.py', PROJECT / 'slr_common.py']
PROJECT.mkdir(parents=True, exist_ok=True)
wanted_names = ('extract3d.py', 'slr_common.py')
code_ready = all(p.exists() for p in required_code)

def download_code_from_shared_folder():
    # 共享文件夹不能通过 /content/drive/MyDrive 直接访问，改用 Drive API。
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    import io
    folder_id = CODE_FOLDER_URL.split('/folders/')[1].split('?')[0].strip('/')
    auth.authenticate_user()
    service = build('drive', 'v3')
    found = set()
    def walk(parent_id):
        token = None
        while True:
            response = service.files().list(
                q=f"'{parent_id}' in parents and trashed = false",
                fields='nextPageToken, files(id,name,mimeType)',
                pageToken=token, pageSize=1000,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            for item in response.get('files', []):
                if item['mimeType'] == 'application/vnd.google-apps.folder':
                    walk(item['id'])
                elif item['name'] in wanted_names:
                    destination = PROJECT / item['name']
                    request = service.files().get_media(fileId=item['id'], supportsAllDrives=True)
                    with open(destination, 'wb') as target:
                        downloader = MediaIoBaseDownload(target, request)
                        done = False
                        while not done:
                            _, done = downloader.next_chunk()
                    found.add(item['name'])
                    print('downloaded code from shared Drive:', item['name'])
            token = response.get('nextPageToken')
            if not token:
                break
    walk(folder_id)
    return all(name in found for name in wanted_names)

# 先从用户提供的共享文件夹读取代码。
if not code_ready:
    try:
        code_ready = download_code_from_shared_folder()
    except Exception as exc:
        print('shared Drive code not available:', exc)

# 如果共享文件夹没有代码，再检查 MyDrive。
if not code_ready:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        code_dirs = [DRIVE_CODE_ROOT] if DRIVE_CODE_ROOT.exists() else []
        my_drive = Path('/content/drive/MyDrive')
        if my_drive.exists():
            code_dirs.extend(sorted({p.parent for p in my_drive.rglob('extract3d.py')}))
        for code_dir in code_dirs:
            if all((code_dir / name).exists() for name in wanted_names):
                for name in wanted_names:
                    shutil.copy2(code_dir / name, PROJECT / name)
                code_ready = True
                print('loaded project code from Drive:', code_dir)
                break
        if not code_ready:
            print('MyDrive 中没有同时找到 extract3d.py 和 slr_common.py')
    except Exception as exc:
        print('Drive code not available:', exc)

# Drive 没有代码时，才从 recognition 分支下载；不影响 Drive 中的视频数据。
if not code_ready:
    repo = 'randlyoyo/FIT5120-TE38-SignLanguage'
    archive_urls = [
        f'https://github.com/{repo}/archive/refs/heads/recognition.zip',
        f'https://codeload.github.com/{repo}/zip/refs/heads/recognition',
    ]
    download_errors = []
    for attempt, archive_url in enumerate(archive_urls):
        archive_path = Path(f'/content/signtest_source_{attempt}.zip')
        try:
            print('downloading project source:', archive_url)
            request = urllib.request.Request(archive_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(request) as response, open(archive_path, 'wb') as target:
                shutil.copyfileobj(response, target)
            with zipfile.ZipFile(archive_path) as archive:
                members = {}
                for member in archive.namelist():
                    filename = Path(member).name
                    if filename in wanted_names and not member.endswith('/'):
                        members.setdefault(filename, member)
                missing = [name for name in wanted_names if name not in members]
                if missing:
                    raise FileNotFoundError('archive does not contain: ' + ', '.join(missing))
                for name in wanted_names:
                    with archive.open(members[name]) as source, open(PROJECT / name, 'wb') as target:
                        shutil.copyfileobj(source, target)
            code_ready = True
            print('project code ready:', PROJECT)
            break
        except Exception as exc:
            download_errors.append(f'{archive_url}: {exc}')
            print('download attempt failed:', exc)
        finally:
            if archive_path.exists():
                archive_path.unlink()
    if not code_ready:
        details = '\n'.join(download_errors)
        raise RuntimeError('Drive 中没有代码，且无法从 recognition 分支下载。\n' + details)

# 先尝试公开链接下载；如果链接需要登录，自动切换到 Google Drive API。
def download_private_drive_folder():
    try:
        from google.colab import auth
    except ImportError as exc:
        raise RuntimeError('当前不是 Colab 环境；请在 Colab 运行，或先把 Drive 文件夹下载到 DATA_ROOT。') from exc
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    import io
    folder_id = DRIVE_FOLDER_URL.split('/folders/')[1].split('?')[0].strip('/')
    service = build('drive', 'v3')
    # 只下载 3D 提取需要的文件；跳过 xlsx/txt/readme 等标注和说明文件。
    download_suffixes = {'.zip', '.mp4', '.mov', '.avi', '.mkv', '.webm', '.task'}
    def walk(parent_id, local_dir):
        local_dir.mkdir(parents=True, exist_ok=True)
        token = None
        while True:
            response = service.files().list(
                q=f"'{parent_id}' in parents and trashed = false",
                fields='nextPageToken, files(id,name,mimeType,size)',
                pageToken=token, pageSize=1000,
                supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
            for item in response.get('files', []):
                dst = local_dir / item['name']
                if item['mimeType'] == 'application/vnd.google-apps.folder':
                    walk(item['id'], dst)
                    continue
                if Path(item['name']).suffix.lower() not in download_suffixes:
                    print('skip non-video:', item['name'])
                    continue
                if dst.exists() and dst.stat().st_size > 0:
                    continue
                request = service.files().get_media(fileId=item['id'], supportsAllDrives=True)
                with open(dst, 'wb') as fh:
                    downloader = MediaIoBaseDownload(fh, request, chunksize=32 * 1024 * 1024)
                    done = False
                    while not done:
                        status, done = downloader.next_chunk()
                        if status:
                            print(item['name'], f'{status.progress() * 100:.1f}%')
            token = response.get('nextPageToken')
            if not token:
                break
    walk(folder_id, DATA_ROOT)

if not DATA_ROOT.exists() or not any(DATA_ROOT.rglob('*')):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    cmd = ['gdown', '--folder', DRIVE_FOLDER_URL, '-O', str(DATA_ROOT), '--remaining-ok']
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        print('gdown 需要公开权限，改用 Colab 登录态下载……')
        download_private_drive_folder()
if not any(DATA_ROOT.rglob('*')):
    raise RuntimeError('没有下载到视频。请确认当前 Colab 账号有权访问这个 Drive 文件夹。')

# 模型优先从项目或 Drive 数据文件夹寻找；找不到时自动下载官方模型。
model_candidates = [PROJECT / 'holistic_landmarker.task']
model_candidates += sorted(DATA_ROOT.rglob('*.task'))
model_candidates = [p for p in model_candidates if p.exists()]
if not model_candidates:
    import urllib.request
    model_url = 'https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task'
    downloaded_model = Path('/content/holistic_landmarker.task')
    print('Drive 中没有模型，自动下载 holistic_landmarker.task ...')
    urllib.request.urlretrieve(model_url, downloaded_model)
    model_candidates = [downloaded_model]
MODEL = model_candidates[0]
print('MODEL:', MODEL)
print('DATA files:', sum(1 for _ in DATA_ROOT.rglob('*')))

## 整理输入视频

项目的输出目录必须保持 `keypoints3d/Train`、`keypoints3d/Valid`、`keypoints3d/Test_STU` 等 split。
下面会自动识别 zip；如果是散装视频，则用 `renders/takes.json` 按文件名归类。

In [ ]:
import json
import os
import shutil
import zipfile
from pathlib import Path

KNOWN_SPLITS = ['Valid', 'Train', 'Test_STU', 'Test_ITW', 'Test_TED', 'Test_SYN', 'Test_MTV']
VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
CLIPS = Path('/content/selected_clips')
CLIPS.mkdir(parents=True, exist_ok=True)

# 完整 zip 模式不依赖 takes.json；只有指定单词或散装视频才需要它。
takes_path = PROJECT / 'renders' / 'takes.json'
takes = json.load(open(takes_path)) if takes_path.exists() else {}
if not takes:
    print('takes.json not found: full zip mode is still supported.')
stem_to_split = {row[0]['stem']: row[0]['split'] for row in takes.values()}

if ONLY_WORDS is not None:
    if not takes:
        raise FileNotFoundError('ONLY_WORDS requires renders/takes.json.')
    unknown = sorted(set(ONLY_WORDS) - set(takes))
    if unknown:
        raise ValueError('Unknown word(s): ' + ', '.join(unknown))
    wanted_stems = {takes[w][0]['stem']: takes[w][0]['split'] for w in ONLY_WORDS}
else:
    wanted_stems = None

# 如果 zip 文件名/目录名没有 split，可以在这里手动指定关键词。
SPLIT_OVERRIDES = globals().get('SPLIT_OVERRIDES', {})
def infer_split(path):
    text = str(path).replace('\\', '/').lower()
    for marker, split in SPLIT_OVERRIDES.items():
        if marker.lower() in text:
            return split
    for split in KNOWN_SPLITS:
        if split.lower() in text:
            return split
    return None

zips = sorted(p for p in DATA_ROOT.rglob('*.zip') if p.is_file())
videos = sorted(p for p in DATA_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
zip_jobs = []
unclassified_zips = []

if zips and wanted_stems is None:
    # 完整数据：直接把每个带 split 的 zip 交给 extract3d.py，支持断点续跑。
    for archive in zips:
        split = infer_split(archive)
        if split:
            zip_jobs.append((split, archive))
        else:
            unclassified_zips.append(archive)
elif zips and wanted_stems is not None:
    # 指定单词：只从 zip 中抽取对应视频，避免解压整套数据。
    for archive in zips:
        with zipfile.ZipFile(archive) as zf:
            for name in zf.namelist():
                item = Path(name)
                stem = item.stem
                if stem not in wanted_stems or item.suffix.lower() not in VIDEO_EXTS:
                    continue
                dst = CLIPS / wanted_stems[stem] / item.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                if not dst.exists() or dst.stat().st_size == 0:
                    with zf.open(name) as src, open(dst, 'wb') as out:
                        shutil.copyfileobj(src, out)
elif videos:
    # 散装视频：优先使用 takes.json 的 stem 映射，否则使用父目录名识别 split。
    for src in videos:
        stem = src.stem
        split = (wanted_stems or {}).get(stem) if wanted_stems is not None else stem_to_split.get(stem)
        split = split or infer_split(src)
        if not split:
            continue
        dst = CLIPS / split / src.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.symlink(src, dst)

clip_dirs = sorted(p for p in CLIPS.iterdir() if p.is_dir())
print('DATA_ROOT:', DATA_ROOT)
print('found zip files:', len(zips), 'video files:', len(videos))
print('zip jobs:', [(split, archive.name) for split, archive in zip_jobs])
if unclassified_zips:
    print('unclassified zip files:', [str(p) for p in unclassified_zips[:20]])
print('prepared clip counts:', {p.name: sum(1 for _ in p.iterdir()) for p in clip_dirs})
if not zip_jobs and not clip_dirs:
    raise RuntimeError('没有可处理的输入。请确认 DATA_ROOT 中有带 Train/Valid/Test_* 路径的视频 zip，或设置 SPLIT_OVERRIDES。')

In [ ]:
from pathlib import Path

root = Path('/content/drive/MyDrive')
files = list(root.rglob('extract3d.py')) + list(root.rglob('slr_common.py'))

for f in files:
    print(f)

In [ ]:
from pathlib import Path
project = Path('/content/signtest')
required_code = [project / 'extract3d.py', project / 'slr_common.py']
if not all(p.exists() for p in required_code):
    if 'download_code_from_shared_folder' not in globals():
        raise RuntimeError('请先运行前面的准备代码 cell。')
    if not download_code_from_shared_folder():
        raise FileNotFoundError('共享 Drive 文件夹中没有同时找到 extract3d.py 和 slr_common.py。')
print('代码已准备好:', project)

In [ ]:
from pathlib import Path
import shutil
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

project = Path('/content/signtest')
code_dir = Path('/content/drive/MyDrive/signtest_code')
project.mkdir(parents=True, exist_ok=True)

for name in ['extract3d.py', 'slr_common.py']:
    src = code_dir / name
    if not src.exists():
        raise FileNotFoundError(f'找不到文件：{src}')
    shutil.copy2(src, project / name)

print('代码已复制：', list(project.iterdir()))

In [ ]:
# A100 GPU 模式会保留 WORKERS 设置；每个 worker 都会创建一个 MediaPipe 实例。
extract_script = PROJECT / 'extract3d.py'
if not extract_script.exists() or not (PROJECT / 'slr_common.py').exists():
    if 'download_code_from_shared_folder' not in globals():
        raise RuntimeError('请先运行前面的准备代码 cell，以读取共享 Drive 文件夹。')
    if not download_code_from_shared_folder():
        raise FileNotFoundError('共享 Drive 文件夹中没有同时找到 extract3d.py 和 slr_common.py。')
    print('copied code from shared Drive folder')

run_workers = WORKERS
if USE_GPU:
    source = extract_script.read_text()
    old = 'base_options=BaseOptions(model_asset_path=model_path),'
    new = 'base_options=BaseOptions(model_asset_path=model_path, delegate=BaseOptions.Delegate.GPU),'
    if old not in source:
        raise RuntimeError('找不到 GPU delegate 插入位置，请检查 extract3d.py 版本')
    # GPU 临时脚本必须和 slr_common.py 放在同一目录，否则子进程无法 import。
    gpu_script = PROJECT / 'extract3d_gpu.py'
    gpu_script.write_text(source.replace(old, new, 1))
    extract_script = gpu_script
    run_workers = WORKERS
print('extract script:', extract_script)
print('workers:', run_workers, 'GPU:', USE_GPU)

In [ ]:
# 先只测试一个视频；GPU/EGL 失败时立即停止，不会把几千个视频全部标成 FAIL。
if USE_GPU and GPU_SMOKE_TEST:
    if not zip_jobs:
        raise RuntimeError('没有 zip 输入，无法进行 GPU 冒烟测试。')
    smoke_zip = zip_jobs[0][1]
    smoke_out = PROJECT / 'gpu_smoke_test'
    smoke_cmd = [sys.executable, str(extract_script), '--zip', str(smoke_zip),
                 '--out', str(smoke_out), '--model', str(MODEL),
                 '--workers', '1', '--limit', '1']
    print('GPU smoke test:', ' '.join(map(str, smoke_cmd)), flush=True)
    smoke_result = subprocess.run(smoke_cmd, text=True, capture_output=True)
    smoke_output = (smoke_result.stdout or '') + (smoke_result.stderr or '')
    print(smoke_output, end='', flush=True)
    if smoke_result.returncode != 0:
        Path('/content/gpu_smoke_test_error.log').write_text(smoke_output, encoding='utf-8')
        raise RuntimeError('GPU/EGL 冒烟测试失败；没有开始全量提取。详见 /content/gpu_smoke_test_error.log')
    print('GPU smoke test passed')

In [ ]:
import subprocess
import sys

def run_extract(args):
    print('RUN:', ' '.join(map(str, args)), flush=True)
    result = subprocess.run(args, text=True, capture_output=True)
    child_output = (result.stdout or '') + (result.stderr or '')
    if child_output:
        print(child_output, end='', flush=True)
    if result.returncode != 0:
        error_log = Path('/content/extract3d_error.log')
        error_log.write_text(child_output, encoding='utf-8')
        tail = child_output[-12000:]
        raise RuntimeError(f'extract3d failed with exit code {result.returncode}\n\nCHILD PROCESS OUTPUT:\n{tail}\n\nFull log: {error_log}')

if zip_jobs:
    for split, archive in zip_jobs:
        run_extract([sys.executable, str(extract_script), '--zip', str(archive),
                     '--out', str(OUTPUT / split), '--model', str(MODEL),
                     '--workers', str(run_workers)])
else:
    if not clip_dirs:
        raise RuntimeError('没有找到可处理的视频。请检查 Drive 权限、目录结构或 ONLY_WORDS。')
    for clip_dir in clip_dirs:
        run_extract([sys.executable, str(extract_script), '--videos', str(clip_dir),
                     '--out', str(OUTPUT / clip_dir.name), '--model', str(MODEL),
                     '--workers', str(run_workers)])
print('3D extraction finished:', OUTPUT)

In [ ]:
import numpy as np

required = {'pose_world', 'left_hand_world', 'right_hand_world', 'pose', 'left_hand', 'right_hand', 'face', 'meta'}
outputs = sorted(OUTPUT.rglob('*.npz'))
bad = []
for path in outputs:
    try:
        with np.load(path, allow_pickle=False) as data:
            missing = required - set(data.files)
            if missing:
                bad.append((str(path), sorted(missing)))
    except Exception as exc:
        bad.append((str(path), str(exc)))
print('valid npz:', len(outputs), 'bad:', len(bad))
if bad:
    print(*bad[:10], sep='\n')
    raise RuntimeError('部分 3D 文件校验失败')

## 保存结果

推荐保存到 Google Drive，避免 Colab 会话结束后丢失。保存后，把 `keypoints3d` 放回本地项目根目录即可。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_TO = Path('/content/drive/MyDrive/signtest/keypoints3d')
shutil.copytree(OUTPUT, SAVE_TO, dirs_exist_ok=True)
print('saved to:', SAVE_TO)

回到本地项目后，确认目录结构为 `keypoints3d/Train`、`keypoints3d/Valid` 等。
`batch_build.py` 会优先读取这些 3D 文件；之后重新生成动画和 GIF 即可。

如果 Drive 分享链接报权限错误，请把文件夹设置为“知道链接的任何人可查看”，或先在 Colab 挂载 Drive 后，把 `DATA_ROOT` 改成实际路径。